In [1]:
"""
TTF Natural Gas — Real-Time Forecast Combinations
===================================================
8 combinations per horizon (72 total): 4 selection methods x 2 weighting schemes.

Selection:  All 26 models | Ratio<1 (recursive) | Highest p-val recursive | Highest p-val rolling 24m
Weighting:  Equal weights | Inverse-MSPE weights

Key decisions:
  - Real-time constraint: selection/weighting use only origins strictly before t
  - MIN_OBS=12: fallback to equal-weighted all-26 for first 12 origins
  - Ratio<1: model kept if recursive MSPE < benchmark recursive MSPE
  - DM test: two-sided, Newey-West HAC, bandwidth=floor(h^(1/3))
  - Sign check: model must beat benchmark on average before DM test runs
  - P-value threshold: p < 0.25
  - N=0 fallback: equal-weighted all-26. N=1: single model used alone.
  - Inv-MSPE weights normalised to sum to 1.
"""

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────────────
INPUT_FILE  = 'Input_All_Forecasts_Combined.xlsx'
OUTPUT_FILE = 'Combination_Forecasts_Output.xlsx'
BENCHMARK   = 'RW_avg (benchmark)'
HORIZONS    = [1, 3, 6, 9, 12, 15, 18, 21, 24]
MIN_OBS     = 12    # minimum past obs before selection/weighting is applied
ROLLING_WIN = 24    # rolling window length for methods 7 & 8
PVAL_THRESHOLD = 0.25  # keep models with DM p-value < 0.25 (mirrors MCS alpha)


# ── DM test (vectorised, Newey-West, bandwidth = floor(h^(1/3))) ──────────────
def dm_pvals_vec(se_models, se_bench, h):
    """
    Compute DM test p-values for M models simultaneously.

    Parameters
    ----------
    se_models : ndarray (T, M) — squared errors for M candidate models
    se_bench  : ndarray (T,)   — benchmark squared errors (same origins)
    h         : int            — forecast horizon (determines bandwidth)

    Returns
    -------
    pvals : ndarray (M,) — two-sided p-values; nan if sign check fails or
                           insufficient valid observations
    """
    T, M = se_models.shape
    bw   = int(np.floor(h ** (1 / 3)))   # Newey-West bandwidth (A06)
    pvals = np.full(M, np.nan)

    for j in range(M):
        m_se = se_models[:, j]

        # Drop rows where either series is nan
        valid = ~(np.isnan(m_se) | np.isnan(se_bench))
        if valid.sum() < MIN_OBS:
            continue

        m_se_v = m_se[valid]
        b_se_v = se_bench[valid]

        # Sign check: model must beat benchmark on average (A07)
        if np.mean(m_se_v) >= np.mean(b_se_v):
            continue

        # Loss differential
        d      = m_se_v - b_se_v
        Tv     = len(d)
        d_mean = np.mean(d)

        # Newey-West long-run variance
        lrv = np.var(d, ddof=0)
        for lag in range(1, bw + 1):
            if lag >= Tv:
                break
            gamma = np.mean((d[lag:] - d_mean) * (d[:-lag] - d_mean))
            lrv  += 2.0 * (1.0 - lag / (bw + 1)) * gamma

        if lrv <= 0:
            continue

        dm_stat  = d_mean / np.sqrt(lrv / Tv)
        pvals[j] = 2.0 * (1.0 - stats.norm.cdf(abs(dm_stat)))

    return pvals


# ── Load data ──────────────────────────────────────────────────────────────────
print("Loading data...")
df = pd.read_excel(INPUT_FILE)
df['forecast_origin'] = pd.to_datetime(df['forecast_origin'])
df['sq_error'] = (df['forecast'] - df['actual']) ** 2

n_candidates = df[df['model'] != BENCHMARK]['model'].nunique()
print(f"  Benchmark : {BENCHMARK}")
print(f"  Candidates: {n_candidates} models")
print(f"  Horizons  : {HORIZONS}")


# ── Main combination loop ─────────────────────────────────────────────────────
all_results = []

for h in HORIZONS:
    print(f"  Processing h={h}...", end='', flush=True)

    # Subset and sort by origin
    sub    = df[df['horizon'] == h].sort_values('forecast_origin')
    bdf    = sub[sub['model'] == BENCHMARK].set_index('forecast_origin')
    cdf    = sub[sub['model'] != BENCHMARK]
    origins = sorted(sub['forecast_origin'].unique())

    # Wide matrices: rows = forecast origins, columns = candidate models
    SE = (cdf.pivot_table(index='forecast_origin', columns='model',
                          values='sq_error', aggfunc='first')
             .reindex(origins))
    FC = (cdf.pivot_table(index='forecast_origin', columns='model',
                          values='forecast', aggfunc='first')
             .reindex(origins))

    B_se    = bdf['sq_error'].reindex(origins).values   # (T,) benchmark sq errors
    B_fc    = bdf['forecast'].reindex(origins).values   # (T,) benchmark forecasts
    ACT     = bdf['actual'].reindex(origins).values     # (T,) realised values

    SE_arr  = SE.values   # (T, M)
    FC_arr  = FC.values   # (T, M)
    M       = SE_arr.shape[1]
    T       = len(origins)

    # ── Origin-level loop ─────────────────────────────────────────────────────
    for ti in range(T):

        act  = ACT[ti]                      # realised value at t
        fc_t = FC_arr[ti]                   # candidate forecasts at t (M,)

        # Models with valid forecast at t
        valid_mask = ~np.isnan(fc_t)
        vc_idx     = np.where(valid_mask)[0]
        if len(vc_idx) == 0:
            continue

        # Past data (strictly before t) — ensures real-time constraint (A03)
        n_past  = ti                        # number of past origins
        se_past = SE_arr[:ti, :]            # (n_past, M) past squared errors
        b_past  = B_se[:ti]                 # (n_past,)   benchmark past sq errors

        # Recursive MSPE per model and for benchmark (A05, A12)
        b_mspe  = np.nanmean(b_past)           if n_past > 0 else np.nan
        m_mspe  = np.nanmean(se_past, axis=0)  if n_past > 0 else np.full(M, np.nan)

        # ── (A) All models — no selection ─────────────────────────────────────
        sel_all = vc_idx.tolist()

        # ── (B) Ratio < 1 selection (A05) ────────────────────────────────────
        sel_rat = []
        rat_fb  = True
        if n_past >= MIN_OBS and not np.isnan(b_mspe) and b_mspe > 0:
            sel_rat = [i for i in vc_idx
                       if not np.isnan(m_mspe[i]) and m_mspe[i] < b_mspe]
        if len(sel_rat) == 0:           # N=0 fallback (A10)
            sel_rat = sel_all[:]
            rat_fb  = True
        else:
            rat_fb  = False

        # ── (C) Highest p-value recursive (A06-A08) ───────────────────────────
        sel_pr = []
        pr_fb  = True
        if n_past >= MIN_OBS:
            vc_se = se_past[:, vc_idx]          # (n_past, |vc|)
            pv    = dm_pvals_vec(vc_se, b_past, h)
            # Keep only models with p-value below threshold (A08)
            ok    = np.where(~np.isnan(pv) & (pv < PVAL_THRESHOLD))[0]
            if len(ok) > 0:
                # Sort by highest p-value within threshold (A08)
                sorted_ok = ok[np.argsort(-pv[ok])]
                sel_pr    = [vc_idx[j] for j in sorted_ok]
                pr_fb     = False
        if pr_fb:                        # N=0 fallback (A10)
            sel_pr = sel_all[:]

        # ── (D) Highest p-value rolling (A09) ────────────────────────────────
        sel_pl = []
        pl_fb  = True
        if n_past >= MIN_OBS:
            roll_start  = max(0, ti - ROLLING_WIN)
            se_roll     = SE_arr[roll_start:ti, :]   # (win, M)
            b_roll      = B_se[roll_start:ti]        # (win,)
            n_roll      = ti - roll_start
            if n_roll >= MIN_OBS:
                vc_se_roll = se_roll[:, vc_idx]
                pv2        = dm_pvals_vec(vc_se_roll, b_roll, h)
                # Keep only models with p-value below threshold (A08)
                ok2        = np.where(~np.isnan(pv2) & (pv2 < PVAL_THRESHOLD))[0]
                if len(ok2) > 0:
                    sorted_ok2 = ok2[np.argsort(-pv2[ok2])]
                    sel_pl     = [vc_idx[j] for j in sorted_ok2]
                    pl_fb      = False
        if pl_fb:                        # N=0 fallback (A10)
            sel_pl = sel_all[:]

        # ── Weighting and combination ─────────────────────────────────────────
        def make_combination(sel_idx, weighting):
            
            # Further filter to models with non-NaN forecast at t (A15)
            vidx = [i for i in sel_idx if not np.isnan(fc_t[i])]
            if not vidx:
                return np.nan, 0

            n = len(vidx)

            if weighting == 'equal':
                w = np.ones(n) / n                          # A10/A11

            else:  # inverse-MSPE (A12)
                mv = m_mspe[vidx]

                # Check for exact forecasters (MSPE=0) (A13)
                zero_mask = (mv == 0) & ~np.isnan(mv)
                if zero_mask.any():
                    w = np.zeros(n)
                    w[zero_mask] = 1.0 / zero_mask.sum()

                elif np.all(np.isnan(mv)) or np.all(mv <= 0):
                    # No valid past MSPE — fall back to equal weights (A14)
                    w = np.ones(n) / n

                else:
                    inv   = np.where((mv > 0) & ~np.isnan(mv), 1.0 / mv, 0.0)
                    total = inv.sum()
                    w     = inv / total if total > 0 else np.ones(n) / n

            fc_combo = float(np.dot(w, fc_t[vidx]))
            return fc_combo, n

        # ── Produce 8 combinations ────────────────────────────────────────────
        combos = [
            ('(1) All-Equal',        sel_all, 'equal',    False   ),
            ('(2) All-InvMSPE',      sel_all, 'inv_mspe', False   ),
            ('(3) Ratio<1-Equal',    sel_rat, 'equal',    rat_fb  ),
            ('(4) Ratio<1-InvMSPE',  sel_rat, 'inv_mspe', rat_fb  ),
            ('(5) PvalRec-Equal',    sel_pr,  'equal',    pr_fb   ),
            ('(6) PvalRec-InvMSPE',  sel_pr,  'inv_mspe', pr_fb   ),
            ('(7) PvalRoll-Equal',   sel_pl,  'equal',    pl_fb   ),
            ('(8) PvalRoll-InvMSPE', sel_pl,  'inv_mspe', pl_fb   ),
        ]

        for name, sel, wt, fb in combos:
            fc_c, n = make_combination(sel, wt)
            se_c    = ((fc_c - act) ** 2
                       if not (np.isnan(fc_c) or np.isnan(act))
                       else np.nan)
            all_results.append({
                'horizon'        : h,
                'forecast_origin': origins[ti],
                'combination'    : name,
                'n_models'       : n,
                'forecast'       : fc_c,
                'actual'         : act,
                'sq_error'       : se_c,
                'fallback_used'  : fb,
            })

    print(" done")


# ── Compute MSPE ratios ────────────────────────────────────────────────────────
print("Computing MSPE ratios...")
res_df = pd.DataFrame(all_results)

# Benchmark MSPE per horizon (over all origins with valid actual)
bench_mspe = {}
for h in HORIZONS:
    b = (df[(df['model'] == BENCHMARK) & (df['horizon'] == h)]
         .dropna(subset=['actual'])
         .copy())
    b['sq_error'] = (b['forecast'] - b['actual']) ** 2
    bench_mspe[h] = b['sq_error'].mean()

combo_names = sorted(res_df['combination'].unique())
mspe_rows   = []

for h in HORIZONS:
    row  = {'Horizon': h}
    sub_h = res_df[(res_df['horizon'] == h)].dropna(subset=['actual', 'sq_error'])
    for c in combo_names:
        cs      = sub_h[sub_h['combination'] == c]
        row[c]  = (round(cs['sq_error'].mean() / bench_mspe[h], 4)
                   if len(cs) > 0 else np.nan)
    mspe_rows.append(row)

mspe_df = pd.DataFrame(mspe_rows).set_index('Horizon')


# ── Supporting tables ─────────────────────────────────────────────────────────
count_df = (res_df.groupby(['horizon', 'combination'])['n_models']
            .agg(['mean', 'min', 'max'])
            .round(1))

fb_df = (res_df.groupby(['horizon', 'combination'])['fallback_used']
         .mean()
         .mul(100)
         .round(1)
         .rename('fallback_pct'))


# ── Assumptions sheet (from docstring) ───────────────────────────────────────
assumptions = [line.strip() for line in __doc__.split('\n')
               if line.strip() and
               (line.strip()[0] in ('A', '✓') or
                line.strip().startswith('VERIFIED') or
                line.strip().startswith('CORRECTION'))]
adf = pd.DataFrame({'': assumptions})


# ── Write output ──────────────────────────────────────────────────────────────
print(f"Writing to {OUTPUT_FILE}...")
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    mspe_df.to_excel(writer, sheet_name='MSPE_Ratios')
    res_df.to_excel(writer, sheet_name='Combinations_Long', index=False)
    count_df.to_excel(writer, sheet_name='Selection_Counts')
    fb_df.to_excel(writer, sheet_name='Fallback_Frequency')
    adf.to_excel(writer, sheet_name='Assumptions', index=False)

print("\nFinal MSPE Ratios:")
print(mspe_df.to_string())
print("\nDone.")


Loading data...
  Benchmark : RW_avg (benchmark)
  Candidates: 26 models
  Horizons  : [1, 3, 6, 9, 12, 15, 18, 21, 24]
  Processing h=1... done
  Processing h=3... done
  Processing h=6... done
  Processing h=9... done
  Processing h=12... done
  Processing h=15... done
  Processing h=18... done
  Processing h=21... done
  Processing h=24... done
Computing MSPE ratios...
Writing to Combination_Forecasts_Output.xlsx...

Final MSPE Ratios:
         (1) All-Equal  (2) All-InvMSPE  (3) Ratio<1-Equal  (4) Ratio<1-InvMSPE  (5) PvalRec-Equal  (6) PvalRec-InvMSPE  (7) PvalRoll-Equal  (8) PvalRoll-InvMSPE
Horizon                                                                                                                                                          
1               0.9948           0.9770             0.8624               0.8025             0.7063               0.7063              0.6501                0.6446
3               1.1471           1.1082             1.0419             